# Notebook 00 — From raw Q&A to a fine-tuning dataset

**Workshop:** Agentic AI for Scientists — Week 4 (Post-Training & Deployment)
**Block:** dataset prep (25 min)
**Goal:** Turn a public medical Q&A corpus (**MedQuAD**, 47k NIH clinician answers) into a clean, structured **instruction-tuning dataset** — the single artifact every later notebook trains on, evaluates against, and finally deploys.

**The honest truth about fine-tuning:** the model is the easy part. *The dataset is the project.* A fine-tune is only as good as the supervision you build here. So we spend a whole notebook on it.

**What we build:**
1. Load MedQuAD and look at what it actually contains (it ships *weak labels* — UMLS concepts, question focus, question type).
2. Use **Gemini as a teacher / labeler** to extract a structured clinical schema from each answer: `disease`, `symptoms[]`, `treatment[]`, `patient_info`, plus a one-line `answer_summary`. (This *teacher-student* pattern — a strong model labels data a small model then learns to imitate — is how most modern instruction datasets are actually made.)
3. Pack each example into the model's **chat template** as instruction -> structured response.
4. Split into `train` / `validation` / `test` and save JSONL.

> Provider: **Gemini free tier** (`gemini-2.5-flash`) for labeling. A full label run over thousands of rows costs time + quota, so we ship a **pre-baked** `data/medquad_structured.jsonl` and only label a few rows live to show the mechanism. Swap in Anthropic by editing one cell if you prefer.

## 0. Setup

In [ ]:
%pip install -q datasets "langchain-google-genai>=2.0" pydantic python-dotenv

In [ ]:
import os

# Free Gemini API key (no credit card): https://aistudio.google.com/apikey
# Colab : add GOOGLE_API_KEY under the key icon (left sidebar) -> "Secrets".
# Local : put GOOGLE_API_KEY=AIza... in a .env file next to this notebook.
try:
    from google.colab import userdata  # type: ignore
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except Exception:
        pass

# Accept GEMINI_API_KEY as an alias (some AI Studio users store it under that name).
if not os.environ.get("GOOGLE_API_KEY") and os.environ.get("GEMINI_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = os.environ["GEMINI_API_KEY"]

assert os.environ.get("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY first (see the comment above)."
print("Gemini API key loaded.")

---
## 1. Load MedQuAD and look at it

MedQuAD = *Medical Question Answering Dataset*: ~47k question/answer pairs scraped from 12 NIH/NLM websites (cancer.gov, GARD, MedlinePlus, ...), each answer written or vetted by clinicians. Crucially it ships **weak structured labels** we can lean on:

- `question_focus` — the medical entity the question is about (e.g. *Hashimoto's Disease*)
- `question_type`  — one of ~16 intents (symptoms, treatment, causes, ...)
- `umls_semantic_group` — the UMLS concept category (Disorders, Chemicals & Drugs, ...)

We'll use these later as a *partial ground truth* to sanity-check our LLM labels.

In [ ]:
from datasets import load_dataset

raw = load_dataset("lavita/MedQuAD", split="train")
print(raw)
print("\ncolumns:", raw.column_names)

In [ ]:
# Normalize column names so this notebook also works on the keivalya/MedQuad mirror
# (which uses Question / Answer / qtype). We standardize on lowercase keys.
def _norm(row):
    q = row.get("question") or row.get("Question") or ""
    a = row.get("answer") or row.get("Answer") or ""
    return {
        "question": (q or "").strip(),
        "answer": (a or "").strip(),
        "focus": (row.get("question_focus") or "").strip(),
        "qtype": (row.get("question_type") or row.get("qtype") or "").strip(),
        "semantic_group": (row.get("umls_semantic_group") or "").strip(),
    }

ds = raw.map(_norm, remove_columns=[c for c in raw.column_names if c not in {"question", "answer"}])
# MedQuAD has some empty answers (a few sources were removed for licensing) — drop them.
ds = ds.filter(lambda r: len(r["answer"]) > 30 and len(r["question"]) > 10)
print(f"Usable rows after cleaning: {len(ds):,}")

ex = ds[0]
print("\nQUESTION :", ex["question"])
print("FOCUS    :", ex["focus"], "| TYPE:", ex["qtype"], "| GROUP:", ex["semantic_group"])
print("ANSWER   :", ex["answer"][:400], "...")

In [ ]:
# Question-type distribution — tells us what behaviors a fine-tune could learn.
from collections import Counter
print("Top question types:")
for t, n in Counter(ds["qtype"]).most_common(12):
    print(f"  {n:6,}  {t or '(none)'}")

---
## 2. Define the target schema

We want the fine-tuned model to read a patient-style medical question and return a **structured clinical extraction** plus a concise answer — not a wall of prose. A tight schema makes the model useful to downstream agents *and* makes evaluation in Week 5 crisp (we can score each field).

We declare it as a Pydantic model so we can ask Gemini for **structured output** (guaranteed-parseable JSON, no regex).

In [ ]:
from pydantic import BaseModel, Field
from typing import List


class ClinicalExtraction(BaseModel):
    """Structured clinical fields extracted from a medical Q&A pair."""
    disease: str = Field(description="The primary disease/condition the question is about. '' if none.")
    patient_info: str = Field(description="Any patient demographics/context implied (age group, sex, risk factors). '' if none.")
    symptoms: List[str] = Field(default_factory=list, description="Symptoms / signs mentioned. [] if none.")
    treatment: List[str] = Field(default_factory=list, description="Treatments, drugs, or interventions mentioned. [] if none.")
    answer_summary: str = Field(description="A faithful 1-2 sentence summary of the answer. No new facts.")

## 3. Build the labeler (teacher model)

The labeler is a single Gemini call with the schema bound to it. `with_structured_output` makes the model fill the Pydantic object directly. We keep the temperature at 0 so labels are deterministic and reproducible.

**Why a strong model labels for a weak one:** the 0.6B student can't reliably produce this schema on its own — yet. By training it on thousands of teacher-made examples, it *learns the behavior*. That is the entire idea behind instruction tuning datasets like Alpaca, OpenOrca, and the medical sets used to build clinical assistants.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

teacher = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
labeler = teacher.with_structured_output(ClinicalExtraction)

LABEL_SYSTEM = (
    "You are a clinical information extractor. Given a medical question and its "
    "reference answer, extract ONLY facts present in the text. Never invent a diagnosis, "
    "drug, or symptom that is not stated. If a field has no information, leave it empty."
)

def label_one(question: str, answer: str) -> ClinicalExtraction:
    msg = (
        f"{LABEL_SYSTEM}\n\nQUESTION:\n{question}\n\nREFERENCE ANSWER:\n{answer[:3000]}"
    )
    return labeler.invoke(msg)

# Live demo on one row — watch a strong model turn prose into structure.
demo = ds[0]
extraction = label_one(demo["question"], demo["answer"])
print("FOCUS (weak label):", demo["focus"])
print(extraction.model_dump_json(indent=2))

In [ ]:
# Sanity check the teacher against MedQuAD's own weak label:
# does the extracted `disease` align with the dataset's `focus`?
focus = demo["focus"].lower()
disease = extraction.disease.lower()
print(f"weak focus : {demo['focus']}")
print(f"teacher    : {extraction.disease}")
print("overlap    :", any(w in disease for w in focus.split()) or any(w in focus for w in disease.split()))

---
## 4. Pack into the chat template

A fine-tuning example is a **conversation**. We use three turns:

- **system** — the role we want the deployed model to adopt
- **user** — the medical question (what a scientist/clinician would type)
- **assistant** — the structured JSON answer (what we want the model to produce)

We render it with the model's *own* `apply_chat_template` so the special tokens (`<|im_start|>`, `<|im_end|>` for Qwen3) match exactly what the model expects at train and inference time. Get this wrong and the model never learns the turn boundaries.

In [ ]:
%pip install -q transformers
from transformers import AutoTokenizer
import json

tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")

SYSTEM_PROMPT = (
    "You are a clinical assistant. Read the medical question and respond with a JSON object "
    "containing: disease, patient_info, symptoms (list), treatment (list), answer_summary."
)

def to_messages(question: str, extraction: ClinicalExtraction) -> list[dict]:
    target = json.dumps(extraction.model_dump(), ensure_ascii=False)
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
        {"role": "assistant", "content": target},
    ]

messages = to_messages(demo["question"], extraction)
rendered = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
print(rendered)

---
## 5. Label a batch (and why we ship a pre-baked file)

Labeling all 47k rows live would burn your free quota and take an hour. In the workshop we:

- label a **small live batch** (e.g. 40 rows) so you see the full pipeline run end-to-end, then
- load the **pre-baked** `data/medquad_structured.jsonl` (already labeled offline) for training.

The loop below is deliberately defensive: it retries, sleeps to respect free-tier rate limits, and skips rows the teacher can't parse. This is what real data pipelines look like.

In [ ]:
import time, json, random
from pathlib import Path

def build_dataset(rows, out_path, sleep=1.5, max_rows=40):
    out = Path(out_path); out.parent.mkdir(parents=True, exist_ok=True)
    written = 0
    with out.open("w") as f:
        for i, r in enumerate(rows):
            if written >= max_rows:
                break
            try:
                ext = label_one(r["question"], r["answer"])
            except Exception as e:
                print(f"  [skip {i}] {type(e).__name__}: {e}")
                time.sleep(sleep); continue
            rec = {
                "messages": to_messages(r["question"], ext),
                "focus": r["focus"], "qtype": r["qtype"],  # keep weak labels for Week 5 eval
            }
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            written += 1
            if written % 10 == 0:
                print(f"  labeled {written} rows...")
            time.sleep(sleep)
    print(f"Wrote {written} rows -> {out}")
    return written

# Live demo batch (small). Shuffle for variety.
sample = ds.shuffle(seed=3407).select(range(80))
build_dataset(sample, "data/medquad_live_demo.jsonl", max_rows=40)

In [ ]:
# --- Pre-baked path (use this for training) ---------------------------------
# If you have the repo's pre-baked file, point at it. Otherwise this cell
# upgrades the live demo into a usable (tiny) dataset so the notebook is
# self-contained even with zero pre-baked data.
import os, json
from pathlib import Path

PREBAKED = Path("data/medquad_structured.jsonl")   # ship this in the repo (offline-labeled)
LIVE     = Path("data/medquad_live_demo.jsonl")
SOURCE   = PREBAKED if PREBAKED.exists() else LIVE
print(f"Using: {SOURCE}  (pre-baked found: {PREBAKED.exists()})")

records = [json.loads(l) for l in SOURCE.read_text().splitlines() if l.strip()]
print(f"Total labeled records: {len(records)}")

In [ ]:
# Train / validation / test split (deterministic).
import random
random.Random(3407).shuffle(records)
n = len(records)
n_test = max(2, int(0.1 * n)); n_val = max(2, int(0.1 * n))
test, val, train = records[:n_test], records[n_test:n_test+n_val], records[n_test+n_val:]

for name, part in [("train", train), ("validation", val), ("test", test)]:
    p = Path(f"data/medquad_{name}.jsonl")
    p.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in part))
    print(f"  {name:11s}: {len(part):4d} rows -> {p}")

---
## 6. (Optional) push to the Hugging Face Hub

Pushing makes the dataset loadable from any Colab with one line — handy so the training notebooks don't depend on local files. Skip if you don't have an HF token.

In [ ]:
# import os
# from huggingface_hub import login
# from datasets import Dataset, DatasetDict
# login(os.environ["HF_TOKEN"])  # add HF_TOKEN to Colab Secrets first
#
# dd = DatasetDict({
#     split: Dataset.from_list([json.loads(l) for l in Path(f"data/medquad_{split}.jsonl").read_text().splitlines() if l.strip()])
#     for split in ["train", "validation", "test"]
# })
# dd.push_to_hub("YOUR_USERNAME/medquad-structured")  # change the repo id
print("Hub push is optional — uncomment above and set HF_TOKEN to use it.")

---
## What you built

A structured, chat-formatted medical instruction dataset with train/val/test splits — the artifact the next four notebooks consume.

**Next — `01_full_sft.ipynb`:** take `Qwen/Qwen3-0.6B` and fine-tune **100% of its weights** to produce this schema. Then we shrink the cost with LoRA (02) and QLoRA (03), and finally deploy (04).

> **Design choice worth saying out loud:** we made the *answer* a JSON object, not free prose. That turns a fuzzy "is this a good answer?" into a measurable "did it get the disease / symptoms / treatment right?" — which is exactly what makes Week 5's evaluation tractable. Schema design *is* eval design.